# Pandas GroupBy and Vectorization Practice

**Topic:** NumPy and pandas &nbsp;·&nbsp; **Difficulty:** Medium

## The data (created in the next cell)

- `events`: a DataFrame with 150,000 rows and columns:
  - `user_id` (drawn from 2,000 possible labels, not all of which appear, because activity is skewed)
  - `timestamp` (datetime over a 30-day window)
  - `event`: one of `"view"`, `"cart"`, `"buy"`
  - `value`: dollar amount (only nonzero for `buy` events)

## Your task

Compute, with vectorized pandas (no `iterrows`):

```python
answers = {
    "busiest_hour": ...,      # int 0-23: hour of day with the most events
    "top_spender": ...,       # user_id with the largest total buy value
    "conversion_rate": ...,   # float: fraction of users with >= 1 buy event
    "median_events_per_user": ...,  # float: median number of events per user
}
```

## Evaluation

The evaluation cell recomputes the truth and checks all four answers. It does not check how you computed them, so the no-`iterrows` rule is on your honour.


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(19)
n, n_users = 150_000, 2000

user_activity = rng.gamma(1.2, 1.0, n_users)
user_ids = rng.choice(n_users, size=n, p=user_activity / user_activity.sum())
hour_weights = np.array([1,1,1,1,1,2,3,5,7,8,8,7,9,8,7,6,6,7,9,10,9,6,3,2], dtype=float)
hours = rng.choice(24, size=n, p=hour_weights / hour_weights.sum())
days = rng.integers(0, 30, n)
ts = (pd.Timestamp("2026-06-01")
      + pd.to_timedelta(days, unit="D")
      + pd.to_timedelta(hours, unit="h")
      + pd.to_timedelta(rng.integers(0, 3600, n), unit="s"))
event = rng.choice(["view", "cart", "buy"], size=n, p=[0.78, 0.16, 0.06])
value = np.where(event == "buy", np.round(rng.gamma(2.0, 22.0, n), 2), 0.0)

events = pd.DataFrame({
    "user_id": [f"u{u:04d}" for u in user_ids],
    "timestamp": ts,
    "event": event,
    "value": value,
}).sample(frac=1, random_state=4).reset_index(drop=True)

print(f"events shape: {events.shape}")
events.head()


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the `events` DataFrame created above to compute the four
# aggregations, then create the `answers` dict shown in "Your task".
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: run this cell to compare your answers to the truth
# =====================================================================
if "answers" not in dir():
    raise NameError("Define the `answers` dict in the solution cell before running this cell.")

_true_hour = int(events["timestamp"].dt.hour.value_counts().idxmax())
_buys = events[events["event"] == "buy"]
_true_spender = _buys.groupby("user_id")["value"].sum().idxmax()
_true_conv = float(_buys["user_id"].nunique() / events["user_id"].nunique())
_true_med = float(events["user_id"].value_counts().median())

_checks = [
    ("busiest_hour", lambda a: int(a) == _true_hour),
    ("top_spender", lambda a: str(a) == _true_spender),
    ("conversion_rate", lambda a: abs(float(a) - _true_conv) < 0.002),
    ("median_events_per_user", lambda a: abs(float(a) - _true_med) < 1e-9),
]
_ok = 0
print("=" * 52)
for _key, _fn in _checks:
    _has = _key in answers
    _good = _has and _fn(answers[_key])
    _ok += _good
    print(f"  {_key:<24} yours={answers.get(_key, 'MISSING')!r:<14} {'CORRECT' if _good else 'WRONG'}")
print("=" * 52)
print(f"Score: {_ok}/4")
print("PASSED: nice work!" if _ok == 4 else "NOT PASSED yet: check each aggregation.")


## Solution

Each answer is one vectorized pandas expression: `dt.hour` + `value_counts` for the busiest hour, a filtered `groupby(...).sum().idxmax()` for the top spender, `nunique` ratios for conversion, and `value_counts().median()` for the per-user median.

```python
buys = events[events["event"] == "buy"]

answers = {
    "busiest_hour": int(events["timestamp"].dt.hour.value_counts().idxmax()),
    "top_spender": buys.groupby("user_id")["value"].sum().idxmax(),
    "conversion_rate": float(buys["user_id"].nunique() / events["user_id"].nunique()),
    "median_events_per_user": float(events["user_id"].value_counts().median()),
}
print(answers)
```
